In [2]:
import pandas as pd
import numpy as np
import yaml
from glob import glob
import os
from pathlib import Path
from datetime import date, timedelta, datetime
from pandas.tseries.holiday import USFederalHolidayCalendar
from pathlib import Path
import re
import hashlib
import calendar

In [3]:
import warnings
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### Functions that may be useful

### AC Transit

Raw data is stop-route level. To get stop level, we want to group by stop and day type, and add up the ridership across routes for each stop and day type combination.

In [4]:
raw_ac_transit = pd.read_excel("transit_agency_ridership_raw_datasets/ac_transit/Daily_Stop_Totals_By_Route_All_DOW_2508FA_with_511ID_Lat_Lon.xlsx")
raw_ac_transit.head()

,STOP_ID_GTFS,STOP_NAME,STOP_LATITUDE,STOP_LONGITUDE,START_DATE,END_DATE,DAY_TYPE,BOARDINGS,ALIGHTINGS,ROUTE,STOP_ID
0,52246,8th St & Portola Av,37.768814,-122.272992,2025-08-10,2026-01-31,WEEKDAY,8,5,30,100060
1,52246,8th St & Portola Av,37.768814,-122.272992,2025-08-10,2026-01-31,WEEKDAY,0,1,W (119),100060
2,52246,8th St & Portola Av,37.768814,-122.272992,2025-08-10,2026-01-31,WEEKDAY,0,1,663,100060
3,52246,8th St & Portola Av,37.768814,-122.272992,2025-08-10,2026-01-31,SAT,7,5,30,100060
4,52246,8th St & Portola Av,37.768814,-122.272992,2025-08-10,2026-01-31,SUN,5,4,30,100060


In [8]:
t_raw_df = raw_ac_transit.groupby(["STOP_ID_GTFS", "DAY_TYPE"]).size().reset_index(name="num_rows")
t_raw_df[t_raw_df["num_rows"] > 1]

,STOP_ID_GTFS,DAY_TYPE,num_rows
8,50019,WEEKDAY,13
28,50110,SAT,5
29,50110,SUN,5
30,50110,WEEKDAY,9
31,50112,SAT,2
...,...,...,...
12482,59997,SUN,2
12483,59997,WEEKDAY,2
12484,59999,SAT,6
12485,59999,SUN,6


In [11]:
processed_ac_transit = raw_ac_transit.groupby(["STOP_ID_GTFS", "STOP_NAME", "STOP_LATITUDE", "STOP_LONGITUDE", 
                                               "START_DATE", "END_DATE", "DAY_TYPE", "STOP_ID"], as_index=False, dropna=False) \
                                     .agg(avg_boardings = ("BOARDINGS", "sum"),
                                          avg_alightings = ("ALIGHTINGS", "sum"))
processed_ac_transit.head()

,STOP_ID_GTFS,STOP_NAME,STOP_LATITUDE,STOP_LONGITUDE,START_DATE,END_DATE,DAY_TYPE,STOP_ID,avg_boardings,avg_alightings
0,50000,Lakeshore Av & MacArthur Blvd,37.809406,-122.246858,2025-08-10,2026-01-31,SAT,9902320,2,25
1,50000,Lakeshore Av & MacArthur Blvd,37.809406,-122.246858,2025-08-10,2026-01-31,SUN,9902320,1,12
2,50000,Lakeshore Av & MacArthur Blvd,37.809406,-122.246858,2025-08-10,2026-01-31,WEEKDAY,9902320,2,30
3,50006,Salesforce Transit Center Bay 6,37.788180,-122.397909,2025-08-10,2026-01-31,WEEKDAY,9904890,78,0
4,50010,Salesforce Transit Center Bay 10,37.788419,-122.397609,2025-08-10,2026-01-31,WEEKDAY,9904930,41,0


In [12]:
t_raw_df = processed_ac_transit.groupby(["STOP_ID_GTFS", "DAY_TYPE"]).size().reset_index(name="num_rows")
t_raw_df[t_raw_df["num_rows"] > 1]

,STOP_ID_GTFS,DAY_TYPE,num_rows


### Unitrans

**Only has trip level data.**

In [8]:
raw_unitrans = pd.read_excel("transit_agency_ridership_raw_datasets/unitrans/Unitrans_FY2024-25_TripLevelAnnualData_CalITP.xlsx", sheet_name='Sheet1')
raw_unitrans.head()

,Date,Start Time (Scheduled),End Time (Scheduled),Out,In,Total,Line (Full),Line (Simplified),Block
0,2024-07-01,06:55:00,07:25:00,2,1.0,3,W - Regular,W,1
1,2024-07-01,07:55:00,08:25:00,NaN,NaN,0,W - Regular,W,1
2,2024-07-01,09:00:00,09:30:00,10,1.0,11,W - Regular,W,1
3,2024-07-01,10:00:00,10:30:00,12,6.0,18,W - Regular,W,1
4,2024-07-01,11:00:00,11:30:00,5,8.0,13,W - Regular,W,1


### Fresno Area Express

Same columns as last year.

Stop id and name fully aligned with GTFS.

In [ ]:
raw_fax = pd.read_excel("transit_agency_ridership_raw_datasets/fresno_area_express/Daily Stop Level Data 7.1.25-6.30.26.xlsx")
raw_fax.head()

In [11]:
raw_fax.groupby(["Date", "StopID"], as_index=False, dropna=False).size().sort_values(by="size", ascending=False)

,Date,StopID,size
553745,2026-06-30,2506,1
0,2025-07-01,5,1
553729,2026-06-30,2487,1
553728,2026-06-30,2486,1
553727,2026-06-30,2485,1
...,...,...,...
6,2025-07-01,11,1
5,2025-07-01,10,1
4,2025-07-01,9,1
3,2025-07-01,8,1


#### OCTA

- Stop-direction-route level
- different cols from last year 
- this year lat/lon provided
- Stop Number align with GTFS (but GTFS has leading zeros)
- Same Stop (Name) can have more than one stop ID and loc (same as and consistent with GTFS)
- Stop Number = 99999 -> Stop Desc = "TRIP FOUND, ROUTE NOT IN POT PATTERNS"

In [12]:
raw_octa = pd.read_excel("transit_agency_ridership_raw_datasets/octa/20260301_20260307_APC_RSM.xlsx")
raw_octa.head()

,APC Date,Day Type,Route Number,Route Desc,Direction,Trip ID,Stop Number,Stop Desc,Stop Latitude,Stop Longitude,Boarding,Alighting
0,2026-03-01,Sunday,1,Long Beach - San Clemente,North,12522690,1501,PACIFIC COAST-DEL OBISPO,33.464959,-117.686544,3,1
1,2026-03-01,Sunday,1,Long Beach - San Clemente,North,12522690,3810,COAST-MOUNTAIN,33.531750,-117.774650,0,2
2,2026-03-01,Sunday,1,Long Beach - San Clemente,North,12522690,3816,COAST-LAGUNA,33.541645,-117.782866,0,1
3,2026-03-01,Sunday,1,Long Beach - San Clemente,North,12522690,5106,NEWPORT TRANS CTR DOCK 5,33.614363,-117.867995,2,2
4,2026-03-01,Sunday,1,Long Beach - San Clemente,North,12522690,6931,EL CAMINO REAL-AVD SAN LUIS REY,33.405434,-117.597681,1,0


In [18]:
raw_octa[["Stop Number", "Stop Desc", "Stop Latitude", "Stop Longitude"]] \
        .drop_duplicates() \
        .groupby(["Stop Number", "Stop Desc"], as_index=False) \
        .size() \
        .sort_values("size", ascending=False)

,Stop Number,Stop Desc,size
4981,99999,"TRIP FOUND, ROUTE NOT IN POT PATTERNS",1
0,2,HASTER-ORANGEWOOD,1
1,3,HASTER-WAKEFIELD,1
2,4,HASTER-KATELLA,1
3,5,ANAHEIM-KATELLA,1
...,...,...,...
10,12,ANAHEIM-WATER,1
9,11,ANAHEIM-SOUTH,1
8,10,ANAHEIM-VERMONT,1
7,9,ANAHEIM-LORRAINE,1


In [19]:
processed_octa = raw_octa.groupby(["APC Date", "Day Type", "Stop Number", "Stop Desc", "Stop Latitude", "Stop Longitude"], as_index=False, dropna=False) \
        .agg(boardings = ("Boarding", "sum"),
             alightings = ("Alighting", "sum"))
processed_octa.head()

,APC Date,Day Type,Stop Number,Stop Desc,Stop Latitude,Stop Longitude,boardings,alightings
0,2026-03-01,Sunday,2,HASTER-ORANGEWOOD,33.795853,-117.906272,32,30
1,2026-03-01,Sunday,3,HASTER-WAKEFIELD,33.799175,-117.906308,44,23
2,2026-03-01,Sunday,4,HASTER-KATELLA,33.802133,-117.906399,14,24
3,2026-03-01,Sunday,5,ANAHEIM-KATELLA,33.804322,-117.906375,64,34
4,2026-03-01,Sunday,6,ANAHEIM-CERRITOS,33.810944,-117.905940,46,87


In [21]:
processed_octa.groupby(["APC Date", "Day Type", "Stop Desc"], as_index=False, dropna=False).size().sort_values(by="size", ascending=False)

,APC Date,Day Type,Stop Desc,size
17432,2026-03-06,Weekday,PACIFIC COAST-1ST,6
11185,2026-03-04,Weekday,PACIFIC COAST-1ST,6
20088,2026-03-07,Saturday,PACIFIC COAST-1ST,6
1939,2026-03-01,Sunday,PACIFIC COAST-1ST,6
4936,2026-03-02,Weekday,PACIFIC COAST-1ST,6
...,...,...,...,...
10,2026-03-01,Sunday,17TH-ENGLISH,1
9,2026-03-01,Sunday,17TH-ENDERLE CENTER,1
6,2026-03-01,Sunday,17TH-CARRIAGE,1
5,2026-03-01,Sunday,17TH-CABRILLO PARK,1


#### SCVTA

- trip-stop-route-direction
- STOP_ID is the GTFS STOP ID
- MAIN_CROSS_STREET is most closed to stop name in GTFS
- Light rail routes included
- **Longitude and Latitude not provided, but provided GTFS, partially aligned Stop ID due to version difference.**

In [27]:
raw_scvta = pd.read_excel("transit_agency_ridership_raw_datasets/scvta/OCT_2025_RBS_FULL_DATA_SET.XLSX")
raw_scvta.head()

,ROUTE_NAME,ROUTE_NUMBER,SERVICE_PERIOD,SERVICE_CODE,DIRECTION_NAME,BRANCH,TRIP_TIME,SORT_ORDER,STOP_ID,MAIN_CROSS_STREET,...,Stop_ID_Num,STOP_DISPLAY,Additional_Notes,PATTERN_KEY,BLOCK,TOTAL_SORT,SORT_SP,ROUTE_REV,SERVICE_CODE2,Stop_ID_REV
0,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,00:35:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,60001.0,1.0,NaN,EB04,3507.0,STATS,2.0,22:,Local,60001:
1,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,01:57:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,60001.0,1.0,NaN,EB04,2707.0,STATS,2.0,22:,Local,60001:
2,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,04:32:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,60001.0,1.0,NaN,EB04,2207.0,STATS,2.0,22:,Local,60001:
3,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,04:58:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,60001.0,1.0,NaN,EB04,2407.0,STATS,2.0,22:,Local,60001:
4,22: Palo Alto - Eastridge,22.0,Saturday,Frequent,EAST,[22]Palo Alto > Eastridge,05:23:00,680.0,1.0,SANTA CLARA TRANSIT CENTER,...,60001.0,1.0,NaN,EB04,2607.0,STATS,2.0,22:,Local,60001:


In [25]:
process_scvta = raw_scvta.groupby(["SERVICE_PERIOD", "SERVICE_CODE", "STOP_ID", "MAIN_CROSS_STREET"], as_index=False, dropna=False) \
                         .agg(avg_boardings = ("AVG_BOARDINGS", "sum"),
                              avg_alightings = ("AVG_ALIGHTINGS", "sum"))
process_scvta.head()

,SERVICE_PERIOD,SERVICE_CODE,STOP_ID,MAIN_CROSS_STREET,avg_boardings,avg_alightings
0,Saturday,Community Bus,111.0,OHLONE-CHYNOWETH STATION,0.333333,0.333333
1,Saturday,Community Bus,616.0,2ND + SANTA CLARA,3.000000,0.000000
2,Saturday,Community Bus,619.0,SAN CARLOS + MARKET (CONVENTION CTR),0.333333,0.333333
3,Saturday,Community Bus,620.0,SAN CARLOS + WOZ,0.000100,0.000100
4,Saturday,Community Bus,750.0,TAMIEN STATION,0.000100,0.000100


#### ACE (Altamont Corridor Express)

- GTFS-RIDE (board_alight.txt provided together with GTFS zip)

In [4]:
raw_ace_ridership = pd.read_csv("transit_agency_ridership_raw_datasets/altamont_corridor_express/board_alight.txt")
raw_ace_ridership.head()

,trip_id,stop_id,stop_sequence,record_use,service_date,boardings,alightings
0,ACE02,STK,0,actual,7/1/2026,0.0,27.0
1,ACE02,LTM,1,actual,7/1/2026,0.0,73.0
2,ACE02,TRC,2,actual,7/1/2026,0.0,53.0
3,ACE02,VAS,3,actual,7/1/2026,17.0,22.0
4,ACE02,LIV,4,actual,7/1/2026,10.0,18.0


In [5]:
raw_ace_stops = pd.read_csv("transit_agency_ridership_raw_datasets/altamont_corridor_express/stops.txt")
raw_ace_stops.head()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,platform_code,tts_stop_name
0,SKT,SKT,Stockton - Robert J Cabral Station,NaN,37.957080,-121.278880,SKT,https://acerail.com/stations/#3411,0,NaN,NaN,1,NaN,Stock-ton — Robert Jay Cab-ral Station
1,LTM,LTM,Lathrop/Manteca Station,NaN,37.798940,-121.264143,LTM,https://acerail.com/stations/#3415,0,NaN,NaN,1,NaN,Lath-rup Man-tee-kuh Station
2,TRC,TRC,Tracy Station,NaN,37.696419,-121.432738,TRC,https://acerail.com/stations/#3420,0,NaN,NaN,1,NaN,Tray-see Station
3,VAS,VAS,Vasco Rd Station,NaN,37.697062,-121.717655,VAS,https://acerail.com/stations/#3419,0,NaN,NaN,1,NaN,Vass-coe Road Station
4,LIV,LIV,Livermore Station,NaN,37.685045,-121.766906,LIV,https://acerail.com/stations/#3421,0,NaN,NaN,1,NaN,Liv-er-more Station


In [20]:
t_processed_ace = pd.merge(raw_ace_ridership, raw_ace_stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
t_processed_ace.head()

,trip_id,stop_id,stop_sequence,record_use,service_date,boardings,alightings,stop_name,stop_lat,stop_lon
0,ACE02,LTM,1,actual,7/1/2026,0.0,73.0,Lathrop/Manteca Station,37.798940,-121.264143
1,ACE02,TRC,2,actual,7/1/2026,0.0,53.0,Tracy Station,37.696419,-121.432738
2,ACE02,VAS,3,actual,7/1/2026,17.0,22.0,Vasco Rd Station,37.697062,-121.717655
3,ACE02,LIV,4,actual,7/1/2026,10.0,18.0,Livermore Station,37.685045,-121.766906
4,ACE02,PLS,5,actual,7/1/2026,16.0,38.0,Pleasanton Station,37.658561,-121.882229


In [22]:
processed_ace = t_processed_ace.groupby(["stop_id", "stop_sequence", "stop_name", "stop_lat", "stop_lon", "service_date"], as_index=False, dropna=False)[["boardings", "alightings"]].sum()

In [23]:
processed_ace.head()

,stop_id,stop_sequence,stop_name,stop_lat,stop_lon,service_date,boardings,alightings
0,FMT,6,Fremont Station,37.559114,-122.007353,7/1/2026,327.0,232.0
1,FMT,6,Fremont Station,37.559114,-122.007353,7/10/2026,304.0,298.0
2,FMT,6,Fremont Station,37.559114,-122.007353,7/11/2026,280.0,307.0
3,FMT,6,Fremont Station,37.559114,-122.007353,7/12/2026,194.0,220.0
4,FMT,6,Fremont Station,37.559114,-122.007353,7/15/2026,257.0,278.0
